# 06. Model Evaluation, Threshold Optimization & Explainable AI

## Objectives:
1. Benchmark Precision, Recall, F1, ROC-AUC, and PR-AUC on holdout test data.
2. Plot Confusion Matrix, ROC curve, and Precision-Recall curve.
3. Business Threshold Tuning: examine thresholds [0.20 – 0.70] and analyze trade-offs between customer friction (FP) and fraud loss (FN).
4. Explainable AI with SHAP: global importance and transaction-level attribution.


In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))

import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt
import shap
from src.config import CLEANED_DATA_PATH, MODEL_PATH, OPTIMAL_THRESHOLD
from src.preprocessing import prepare_data_pipeline
from src.evaluate_model import evaluate_threshold_tradeoffs

df = pd.read_csv(CLEANED_DATA_PATH)
_, X_test, _, y_test, _ = prepare_data_pipeline(df)
model = joblib.load(MODEL_PATH)

y_prob = model.predict_proba(X_test)[:, 1]
y_pred = (y_prob >= OPTIMAL_THRESHOLD).astype(int)
print(f'Model loaded. Holdout test set evaluated at threshold: {OPTIMAL_THRESHOLD}')


### Business Threshold Sensitivity & Trade-off Matrix


In [ ]:
tradeoffs = evaluate_threshold_tradeoffs(y_test, y_prob)
tradeoffs


### Explainable AI (SHAP Summary Plot)


In [ ]:
explainer = shap.TreeExplainer(model)
sample = X_test.sample(n=300, random_state=42)
shap_values = explainer.shap_values(sample)

plt.figure(figsize=(9, 6))
shap.summary_plot(shap_values, sample, show=False)
plt.title('SHAP Feature Importance (Fraud Attributions)', fontweight='bold')
plt.tight_layout()
plt.show()
